# Cleaning 2.2 - Clean unique values in panel dataset

This notebook does the following:
    (1) Cleans unique values of variables using the clean_unique_values function
    (2) Drops all rows to be excluded due to unused equipment or different types of equipment
    (3) Cleans affected variables using the clean_affected_vars function
    (5) Cleans the screens variable when total no given instead of per unit no
    (6) Splits types where needed
    (7) Deals with special cases

## Set-up

In [1]:
# Indicator for new variables to be cleaned (default is False, set to True if decide to clean more variables)
new_vars_to_clean = False

In [2]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from create_empty_cleaning_sheet import create_empty_cleaning_sheet
from unique_values_cleaning import clean_unique_values
from create_empty_aff_vars_sheet import create_empty_aff_vars_sheet
from affected_vars_cleaning import clean_affected_vars
from split_types_cleaning import clean_split_types, reassign_type_no


In [3]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_1.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [4]:
# Load data dictionaries
equipment_data_dict = pd.read_excel(config.DATA_DICTIONARIES / "data_dictionary.xlsx", sheet_name="Equipment")

In [5]:
# Load equipment to exclude list
exclusion_list = pd.read_excel(config.CLEANING_WORKBOOKS / "equipment_to_exclude.xlsx", sheet_name = "Exclusion_List")

In [6]:
# Load equipment special cases list
equipment_special_cases = pd.read_excel(config.CLEANING_WORKBOOKS / "equipment_special_cases.xlsx", sheet_name = "Special_Cases")

## (1) Clean unique values of variables

In [7]:
# Create or update the excel workbook with empty sheets
if new_vars_to_clean:

    # File name
    file_name = config.CLEANING_WORKBOOKS / "panel_cleaning_workbook.xlsx"

    # Equipment questions (one sheet per variable)
    for _, row in equipment_data_dict.iterrows():   
        sheet_name = row["Variable"]
        free_text = row["Free text"] == "Y"
        mc_fc_vars = free_text
        create_empty_cleaning_sheet(file_name, sheet_name, 
                                    comment = True, 
                                    free_text = free_text, 
                                    mc_fc_vars = mc_fc_vars)
        
    # Share and EL check variables
    for var in ["share", "el_check"]:
        create_empty_cleaning_sheet(
            file_name, sheet_name=var,
            comment=True, 
            free_text=False,
            mc_fc_vars=False)

In [8]:
# Run cleaning loop. This will: 
# 1. Merge our data to the cleaning workbook
# 2. Create cleaned variables
# 3. produce a report of the cleaning process
# 4. Update the cleaning workbook with all uncleaned values

file_name = config.CLEANING_WORKBOOKS / "panel_cleaning_workbook.xlsx"

# Equipment questions (one sheet per variable)
for _, row in equipment_data_dict.iterrows():
    var_name = row["Variable"]
    sheet_name = var_name
    free_text = row["Free text"] == "Y"
    mc_fc_vars = free_text
    equipment=clean_unique_values(df=equipment, file_name=file_name, 
                                  sheet_name=sheet_name, var_name=var_name, 
                                  comment=True, free_text=free_text, 
                                  mc_fc_vars=mc_fc_vars, 
                                  report=False, 
                                  affected_vars=True)

# Share and EL check variables
for var in ["share", "el_check"]:
    equipment=clean_unique_values(df=equipment, file_name=file_name, 
                                  sheet_name=var, var_name=var, 
                                  comment=True, free_text=False, 
                                  mc_fc_vars=False, report=True,
                                  affected_vars=True)

Cleaning progress for share:
Total unique value combinations: 96
Cleaned combinations: 0
Pending combinations: 0
Excluded combinations: 0
Unchecked combinations: 96
Cleaning progress for el_check:
Total unique value combinations: 59
Cleaned combinations: 0
Pending combinations: 0
Excluded combinations: 0
Unchecked combinations: 59


In [9]:
# For "pending" values, print the labgroupids, survey and variable names and value so that we can check the surveys

# Equipment questions - print labgroupids and survey for pending values
for _, row in equipment_data_dict.iterrows():
    var_name = row["Variable"]
    for s in ["BL", "EL"]:
        for e in ["fc", "fridge", "freezer", "ult", "glassware", "microbio", "cryostat", "bath", "incubator", "heater", "it"]:
            pending_values = equipment[
                (equipment[f"{var_name}_status"] == "Pending") &
                (equipment["survey"] == s) & 
                (equipment["equipment"] == e)
            ]["labgroupid"].tolist()
            if pending_values:
                print(f"Pending values for {var_name}, survey {s}, equipment {e}: {pending_values}")


Pending values for number, survey EL, equipment it: [646, 646]


In [10]:
# For FCs, view all the relevant columns to identify gaps in the data

# List of FC variables (controller_type, sash_width, face_velocity, number, lifted, hours_open, days, surface)
fc_vars = ["controller_type", "sash_width", "face_velocity", "number", "lifted", "hours_open", "days", "surface"]

fc_data = equipment[equipment["equipment"] == "fc"]

# Keep only relevant cols for FC variables
cols_to_keep = ["labgroupid", "survey"] + [f"{var}_clean" for var in fc_vars]
fc_data = fc_data[cols_to_keep]

In [11]:
# For ULTs, view all the relevant columns to identify gaps in the data
ult_data = equipment[equipment["equipment"] == "ult"]
# Keep only relevant cols for ULT variables (ult_type, size_ult, temp_ult, number, seals, spacing, filter, door_openings)
ult_vars = ["ult_type", "size_ult", "temp_ult", "number", "seals", "spacing", "filter", "door_openings"]
cols_to_keep_ult = ["labgroupid", "survey"] + [f"{var}_clean" for var in ult_vars]
ult_data = ult_data[cols_to_keep_ult]

## (2) Drop all rows to be excluded

In [12]:
# If any *_status column is "Exclude", mark the row for exclusion
status_cols = [c for c in equipment.columns if c.endswith("_status")]

equipment["exclude_status"] = (
    equipment[status_cols].eq("Exclude").any(axis=1) if status_cols else False
)

# For each equipment, print the number of rows with "exclude" = True
for eq in equipment["equipment"].unique():
    num_exclude = equipment[(equipment["equipment"] == eq) & (equipment["exclude_status"] == True)].shape[0]
    print(f"Number of rows to exclude based on cleaning workbook for {eq}: {num_exclude}")

# Now count the number of rows to exclude based on the exclusion list

exclusion_list["exclude_list"] = True  # Add an "exclude" column to the exclusion list

# Merge the exclusion list with the equipment data
equipment = equipment.merge(exclusion_list[["labgroupid", "equipment", "type_no", "survey", "exclude_list"]
                                        ], on=["labgroupid", "equipment", "type_no", "survey"], how="left")

# If "exclude" is True in either the cleaning workbook or the exclusion list, mark the row for exclusion
equipment["exclude"] = equipment["exclude_status"] | equipment["exclude_list"]
equipment.drop(columns=["exclude_status", "exclude_list"], inplace=True)

# For each equipment, print the number of rows with "exclude" = True after merging with exclusion list
for eq in equipment["equipment"].unique():
    num_exclude = equipment[(equipment["equipment"] == eq) & (equipment["exclude"] == True)].shape[0]
    print(f"Number of rows to exclude based on cleaning workbook and exclusion list for {eq}: {num_exclude}")

# Drop rows marked for exclusion (and exclude variable)
equipment = equipment[~equipment["exclude"]].drop(columns=["exclude"]).copy()

Number of rows to exclude based on cleaning workbook for fc: 6
Number of rows to exclude based on cleaning workbook for fridge: 7
Number of rows to exclude based on cleaning workbook for freezer: 2
Number of rows to exclude based on cleaning workbook for ult: 16
Number of rows to exclude based on cleaning workbook for glassware: 21
Number of rows to exclude based on cleaning workbook for microbio: 10
Number of rows to exclude based on cleaning workbook for bath: 14
Number of rows to exclude based on cleaning workbook for incubator: 5
Number of rows to exclude based on cleaning workbook for heater: 14
Number of rows to exclude based on cleaning workbook for it: 20
Number of rows to exclude based on cleaning workbook for cryostat: 21
Number of rows to exclude based on cleaning workbook and exclusion list for fc: 10
Number of rows to exclude based on cleaning workbook and exclusion list for fridge: 7
Number of rows to exclude based on cleaning workbook and exclusion list for freezer: 2
Nu

## (3) Deal with affected variables

In [13]:
# Create affected variables cleaning sheet if doesn't already exist
affected_vars_cleaning_file = config.CLEANING_WORKBOOKS / "affected_variables_cleaning.xlsx"

id_cols = ["labgroupid", "equipment", "type_no", "survey"]

create_empty_aff_vars_sheet(
    file_name=affected_vars_cleaning_file,
    sheet_name="Panel Vars",
    id_cols=id_cols
)

Sheet Panel Vars already exists. No changes made.


In [14]:
# Clean affected variables
equipment = clean_affected_vars(
    df=equipment,
    file_name=affected_vars_cleaning_file,
    sheet_name="Panel Vars",
    data_dict = None,
    id_cols=id_cols
)

No new affected variable cases found.
Pulled back cleaned values for 0 affected variable case(s) where value_changed == 'Y'.


## (5) Deal with special cases

In [15]:
# Print the special cases
print("Special cases:")
print(equipment_special_cases)

Special cases:
   labgroupid equipment  type_no survey  \
0         208        it        1     BL   
1         208        it        1     EL   
2         208        it        2     BL   
3         208        it        2     EL   

                                              action  
0  Copy all values from type 1 to type 2 except m...  
1  Copy all values from type 1 to type 2 except m...  
2  Copy all values from type 1 to type 2 except m...  
3  Copy all values from type 1 to type 2 except m...  


In [16]:
# Deal with special cases (currently handled case-by-case)

# Merge special-case actions onto equipment
equipment = equipment.merge(
    equipment_special_cases,
    on=["labgroupid", "equipment", "type_no", "survey"],
    how="left"
)

# Case 1: "Copy all values from type 1 to type 2 except monitor and screen"
# Keep monitor_clean from type_no == 2, but copy all other *_clean values from type_no == 1
action_text = "Copy all values from type 1 to type 2 except monitor and screen"

if "action" in equipment.columns:
    flagged = equipment[
        equipment["action"].astype("string").str.strip() == action_text
    ][["labgroupid", "equipment", "survey"]].drop_duplicates()

    if not flagged.empty:
        clean_cols = [c for c in equipment.columns if c.endswith("_clean") and c != "monitor_clean" and c != "screens_clean"]

        source = equipment[
            equipment["type_no"] == 1
        ][["labgroupid", "equipment", "survey"] + clean_cols].copy()
        source = source.rename(columns={c: f"{c}__src" for c in clean_cols})

        equipment = equipment.merge(
            source,
            on=["labgroupid", "equipment", "survey"],
            how="left"
        )

        flagged_index = flagged.set_index(["labgroupid", "equipment", "survey"]).index
        target_mask = (
            (equipment["type_no"] == 2)
            & equipment.set_index(["labgroupid", "equipment", "survey"]).index.isin(flagged_index)
        )

        for c in clean_cols:
            src_col = f"{c}__src"
            equipment.loc[target_mask, c] = equipment.loc[target_mask, src_col]

        drop_src_cols = [f"{c}__src" for c in clean_cols if f"{c}__src" in equipment.columns]
        if drop_src_cols:
            equipment = equipment.drop(columns=drop_src_cols)

        print(f"Applied special case to {int(target_mask.sum())} type_no==2 row(s).")
    else:
        print("No rows flagged for this exact action text.")
else:
    print(f"Special-case action column not found: 'action'")

Applied special case to 2 type_no==2 row(s).


## (6) Clean the screens variable

In [17]:
# If screens_status is "Total", create screens_clean_per_unit by dividing screens by number
equipment.loc[equipment["screens_status"] == "Total", 
                      "screens_clean_per_unit"] = equipment["screens_clean"] / equipment["number_clean"]

## (7) Split types where needed

In [18]:
# Export/import split-type rows via cleaning workbook
split_types_cleaning_file = config.CLEANING_WORKBOOKS / "split_types_cleaning.xlsx"

split_id_cols = ["labgroupid", "equipment", "type_no", "survey"]
split_clean_cols = [c for c in equipment.columns if c.endswith("_clean")]

equipment = clean_split_types(
    df=equipment,
    file_name=split_types_cleaning_file,
    sheet_name="Panel Split Types",
    id_cols=split_id_cols,
    clean_cols=split_clean_cols,
    split_status="Split",
    auto_assign_type_no=True,
    split_type_start=11,
)


No new Original rows to append.
No Split rows found in workbook yet. Export completed.


## Save processed data

In [19]:
# Final renumbering of type_no after exclusions, split handling, and special cases
equipment = reassign_type_no(
    equipment,
    group_cols=("labgroupid", "equipment", "survey"),
    type_col="type_no",
    start=1,
)

In [20]:
# Save processed data
equipment.to_csv(config.PROCESSED_DATA / "panel_processed_2.csv", index =False)